In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyECLAT import ECLAT
from mlxtend.preprocessing import TransactionEncoder

In [ ]:
#loading dataset
df=pd.read_csv('Market_Basket_Optimisation.csv', header=None)
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
1,burgers,meatballs,eggs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chutney,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,turkey,avocado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mineral water,milk,energy bar,whole wheat rice,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#creating transactions list
transactions=[]
for i in range(0,7501):
    transactions.append([str(df.values[i, j]) for j in range(0,20)])

In [ ]:
#ECLAT model train
eclat = ECLAT(data=df)

# run algorithm
indices, support = eclat.fit(
    min_support=0.05,
    min_combination=2,
    max_combination=2
)

Combination 2 by 2


300it [00:01, 204.13it/s]


In [54]:
support.items()

dict_items([('eggs & mineral water', 0.05092654312758299), ('spaghetti & mineral water', 0.05972536995067324), ('chocolate & mineral water', 0.05265964538061592)])

In [52]:
df_rules=pd.DataFrame(list(support.items()), columns=['itemsets', 'support']).sort_values(by=['support'], ascending=False)
df_rules

,itemsets,support
1,spaghetti & mineral water,0.059725
2,chocolate & mineral water,0.052660
0,eggs & mineral water,0.050927


In [49]:
from mlxtend.frequent_patterns import association_rules
rules=association_rules(df_rules, metric='confidence', min_threshold=0.02, support_only=True)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,"frozenset({n, w, e, , i, r, a, t, l})",frozenset({m}),NaN,NaN,0.238368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"frozenset({n, w, e, , i, r, a, t, m})",frozenset({l}),NaN,NaN,0.238368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"frozenset({n, w, e, , i, r, a, l, m})",frozenset({t}),NaN,NaN,0.238368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"frozenset({n, w, e, , i, r, t, l, m})",frozenset({a}),NaN,NaN,0.238368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"frozenset({n, w, e, , i, a, t, l, m})",frozenset({r}),NaN,NaN,0.238368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86971,"frozenset({u, s})","frozenset({o, p})",NaN,NaN,0.050527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86972,frozenset({o}),"frozenset({u, p, s})",NaN,NaN,0.050527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86973,frozenset({p}),"frozenset({o, u, s})",NaN,NaN,0.050527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86974,frozenset({u}),"frozenset({o, p, s})",NaN,NaN,0.050527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# FPGrowth & Association Rules

In [ ]:
te=TransactionEncoder()
df=pd.DataFrame(te.fit_transform(transactions), columns=te.columns_)
df


,asparagus,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,False,True,True,False,True,False,False,False,False,False,...,False,True,False,False,True,False,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,True,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7496,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
7497,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
7498,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
7499,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
from collections import defaultdict
item_to_transactions=defaultdict(set)
item_to_transactions

defaultdict(set, {})

In [ ]:
for tx_id, items in enumerate(transactions):
    for item in items:
        item_to_transactions[item].add(tx_id)
for item, tx_id in item_to_transactions.items():
    print(f"{item}: {tx_id}")

shrimp: {0, 6147, 6149, 4108, 6156, 6158, 16, 2064, 6163, 20, 23, 4120, 2078, 4127, 6175, 4132, 2089, 2096, 2101, 2104, 4152, 4154, 69, 4168, 6219, 2124, 2134, 91, 107, 108, 2155, 110, 2159, 6251, 2170, 2172, 125, 6270, 2177, 2178, 6275, 6279, 142, 143, 4244, 149, 2197, 153, 155, 6302, 160, 2208, 164, 4264, 6318, 4274, 181, 2233, 2236, 4288, 2241, 2243, 6341, 199, 2252, 2261, 4317, 222, 4321, 4325, 236, 2292, 4343, 2297, 6400, 2308, 4359, 6409, 269, 2319, 273, 4377, 2334, 290, 6438, 6439, 6446, 2356, 2358, 4407, 2365, 6468, 2373, 328, 2381, 2383, 6486, 4440, 2393, 6492, 349, 4452, 2409, 373, 4471, 6522, 380, 2429, 6529, 390, 4493, 6543, 6553, 2480, 433, 6577, 2485, 6582, 6589, 4543, 6595, 6597, 455, 6600, 460, 461, 2509, 468, 469, 471, 474, 477, 479, 4582, 6630, 6643, 503, 505, 507, 6655, 6673, 6684, 6697, 6702, 6704, 565, 2615, 2617, 2618, 6715, 6716, 6717, 6719, 2632, 2635, 2637, 6749, 4704, 610, 4709, 4711, 2667, 620, 2669, 6766, 6775, 633, 2694, 2695, 6793, 656, 659, 662, 664, 4760

In [ ]:
from mlxtend.frequent_patterns import fpgrowth

frequent_itemsets = fpgrowth( df, min_support=0.01, use_colnames=True)

print(frequent_itemsets.head())

    support          itemsets
0  0.238368   (mineral water)
1  0.132116       (green tea)
2  0.076523  (low fat yogurt)
3  0.071457          (shrimp)
4  0.065858       (olive oil)


In [ ]:
#frequent_itemsets.sort_values('support', ascending=True)

In [ ]:
from mlxtend.frequent_patterns import association_rules
rules=association_rules(frequent_itemsets, metric='confidence', min_threshold=0.02)
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

,antecedents,consequents,support,confidence,lift
0,(nan),(mineral water),0.238235,0.238267,0.999574
1,(mineral water),(nan),0.238235,0.999441,0.999574
2,(green tea),(mineral water),0.031063,0.235116,0.986357
3,(mineral water),(green tea),0.031063,0.130313,0.986357
4,(nan),(green tea),0.131982,0.132000,0.999124
...,...,...,...,...,...
1652,(mushroom cream sauce),(nan),0.019064,1.000000,1.000133
1653,(nonfat milk),(nan),0.010399,1.000000,1.000133
1654,(eggplant),(nan),0.013198,1.000000,1.000133
1655,(fromage blanc),(nan),0.013598,1.000000,1.000133


In [ ]:
top_itemsets=frequent_itemsets.head(5)
top_itemsets

,support,itemsets
0,0.238368,(mineral water)
1,0.132116,(green tea)
2,0.076523,(low fat yogurt)
3,0.071457,(shrimp)
4,0.065858,(olive oil)
